# Let's build GPT — a decoder-only transformer
*Karpathy, Neural Networks: Zero to Hero #7*

Building a character-level language model on the tiny-Shakespeare corpus: a bigram baseline first, then the "math trick" behind self-attention, a single attention head, scaling, and LayerNorm — finally assembling the complete transformer (`bigram-gpt.py`) at the end.

Original code and inline comments are preserved verbatim (they're the thinking); markdown adds structure, answers the open `# why do we need this?` questions, and flags a couple of mislabeled comments.

## 1. Data and tokenization
Read the corpus, then build a **character-level tokenizer** — a lossless, reversible `text ↔ list[int]` mapping (`decode(encode(s)) == s`). Production models use *subword* tokenizers instead (Google's SentencePiece, OpenAI's tiktoken/BPE): shorter sequences at the cost of a larger vocabulary.

In [ ]:
# read it in to inspect it
with open('../data/gpt-input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
print("length of dataset in characters: ", len(text))
print(text[:100])

In [ ]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

In [ ]:
# create a mapping from characters to integers
# this is a simple way but google has sentence peice  whcih is a subword tokenizer or openai using tiktoken which is a byte pair tokenizer

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

In [ ]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

## 2. Train / val split
First 90% trains, last 10% validates.

In [ ]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

## 3. Context and targets
`block_size` is the maximum context length. A chunk of `block_size + 1` characters holds `block_size` training examples at once — predict char `t+1` from chars `≤ t` — so the model learns to predict from as little as 1 token of context up to the full block.

In [ ]:
block_size = 8
train_data[:block_size+1]

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

## 4. Batching — the `(B, T)` tensor
Stack several random chunks into a batch. Inputs `xb` and targets `yb` are both `(batch_size, block_size)`, with `yb` equal to `xb` shifted left by one. Every `(b, t)` position is one next-char prediction, so a `(4, 8)` batch packs **32** examples.

In [ ]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(ix, data[ix[0]:ix[0]+block_size], decode(data[ix[0]:ix[0]+block_size].tolist()))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

In [ ]:
print(xb) # our input to the transformer

## 5. Bigram baseline
The simplest model: an embedding table of shape `(vocab, vocab)` whose row for a token *is* the logits for the next token — no context beyond the current character. Loss is cross-entropy; `generate` samples one token at a time and appends it. Expected loss at init ≈ `−ln(1/65) ≈ 4.17`.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # batch by time by channel, here channel is the vocab_size, btach is 4
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx) # goes to the forward fucntion 
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist())) # idx is the first context we give 


## 6. Training loop
AdamW: sample a batch → forward → zero grads → backward → step. The bigram model bottoms out around ~2.4–2.5 loss; with no way to use context, that's its ceiling until we add attention.

In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(10000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist()))

## 7. The math trick behind self-attention
Before real attention, build up its core operation: each token aggregating information from the tokens **before** it. Start with the crudest version — a plain average of the past (loses a lot of information; we fix that with learned weights later).

In [ ]:
# consider the following toy example:
# this is self attention so we have 8 tokens in a batch and they are ot influencing each other 
# we want them to talk to each other for example 5th token should not talk to 6th, 7th or 8th token as those are future
# but rather 5th should only talk to 4th, 3rd, 2nd and 1st. 
# one way is to take average or sum of previous token but we loose information, we will fix this later
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

In [ ]:
# We want x[b,t] = mean_{i<=t} x[b,i] # this is the looking at previous tokens by average
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)
print(x[0])
print(xbow[0])
# notce the first row will be equal and then later ones are average

### Weighted aggregation via matrix multiply
A lower-triangular matrix of **normalized ones**, matrix-multiplied with `x`, computes exactly that running average of the past — vectorizing the double loop. `(T, T) @ (B, T, C)` broadcasts over the batch.

In [ ]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
# a = torch.ones(3, 3) # older version
a = torch.tril(torch.ones(3, 3)) # makes a traingle matrix of ones like a right angle tirangler
a = a / torch.sum(a, 1, keepdim=True) # this line allows us to do average
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

In [ ]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
print(torch.allclose(xbow, xbow2, rtol=1e-04))

### Same average, via masked softmax
Rebuild the same result differently: start from zeros, mask the future to `−inf`, then softmax. The `−inf` entries exponentiate to 0, so you again get an average over the past — but now the **pre-softmax scores can be learned and data-dependent** instead of uniform. That's the doorway to attention.

In [ ]:
# version 3: use Softmax
# in short we can have weighted aggregation of past elements, and when we use it to see how much of each past is imp
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3, rtol=1e-04)

## 8. A single self-attention head
Now the weights become data-dependent. Each token emits a **query** (what I'm looking for), a **key** (what I have), and a **value** (what I'll share if attended to). Affinities `wei = q @ kᵀ` score how much each token wants each other token; mask + softmax; then aggregate the **values** (not the raw `x`). Keeping `value` separate lets a token expose something different from its raw identity.

In [ ]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ x
out.shape

In [ ]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# the idea is each token wll hold a key and query , what i want and what i have
# then if i do dot product of keys i should have high interaction between useful past tokens.
# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T) this is the ddot product we were talking about

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

In [ ]:
wei[0]

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

## 9. Scaling — why divide by √(head_size)
> **Precise version of the last bullet above:** the code does `wei * head_size**-0.5`, i.e. it **divides the scores by √(head_size)** (equivalently, multiplies by `1/√(head_size)`). The note's phrase "divides `wei` by `1/sqrt(head_size)`" reads backwards — dividing *by* `1/√(head_size)` would *multiply* by `√(head_size)`. The intent (and the code) is division by `√(head_size)`.

Without scaling, `q @ kᵀ` has variance ≈ `head_size`, so wide heads produce large scores; softmax of large scores turns **peaky** (converges toward one-hot), attends to a single token, and passes almost no gradient — the same saturation problem as squashing activations. The `head_size**-0.5` factor restores unit variance so softmax stays diffuse early in training. Cells below show softmax getting peaky when the same logits are scaled ×8.

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

In [ ]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5
print(k.var())
print(q.var())
print(wei.var())


## 10. LayerNorm
`LayerNorm1d` is the earlier `BatchNorm1d` with one change: normalize over the **feature** dimension of each row (each token), not over the batch. This removes batch norm's cross-example coupling — a token's normalized output no longer depends on its batch-mates — so train and inference behave identically and no running statistics are needed.

> **Mislabeled-comment flag:** inside `LayerNorm1d`, `xmean = x.mean(1, …)` and `xvar = x.var(1, …)` are commented `# batch mean` / `# batch variance`. For layer norm these are **per-token feature** statistics (reduced over `dim=1` = features), *not* batch statistics — the comments are leftovers from the BatchNorm version. The code is correct; only the comments mislead. Cells 27–28 confirm it: a *feature*'s stats across the batch are not 0/1, but each *row's* (token's) stats are.

In [ ]:
class LayerNorm1d: # (used to be BatchNorm1d) this is used in place of batchnorm this is used to nomalize to gaussian distribution according to rows
  # this removes the input effecting others problem we had in batchnorm

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

In [ ]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

In [ ]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

## 11. The complete model — assembled (`bigram-gpt.py`)
The build above stops at the individual pieces. Here is the full **decoder-only transformer** that puts them together:

`token + positional embeddings → n_layer × Block(masked multi-head attention + feed-forward, each with a residual connection and a pre-block LayerNorm) → final LayerNorm → lm_head → vocab logits`, trained with cross-entropy on next-token prediction.

**Your two open `# why do we need this?` questions, answered:**

- **`self.proj = nn.Linear(n_embd, n_embd)` in `MultiHeadAttention`.** It's the projection back into the **residual stream**. After concatenating the heads you linearly recombine them at the embedding dimension before adding the result to `x`. Without it you'd be adding raw concatenated head outputs straight back — the projection gives the block a learned mix.
- **`nn.Linear(4 * n_embd, n_embd)` in `FeedFoward`.** Same reason: the feed-forward widens to `4·n_embd` to compute, then this second linear **projects back down** to `n_embd` so it fits the residual add. (You already noted the `4×` is the paper's hyperparameter.)

**On your rough note about encoder / decoder / cross-attention:** this model is **decoder-only** — masked self-attention only, with **no encoder and no cross-attention**. Each `Block` is `x = x + sa(ln1(x)); x = x + ffwd(ln2(x))` — the `x + …` is the residual shortcut; `ln1/ln2` are *pre-norm* (applied before each sub-block). The encoder + decoder + cross-attention picture belongs to the *original* translation transformer, not GPT. Full correction is in the notes file.

> **Cosmetic:** the class is spelled `FeedFoward` (missing an `r`) — harmless, since it's consistent everywhere; left verbatim.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# this is decode only transformer this one yet

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 3e-4
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
device = 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('../data/gpt-input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()# this will average our loss
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei) # comes last optimization as things get more deep
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd) # why do we need this ?
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), # multiplicaiton of 4 is new is done to match the papaer just a hyperparameter tuning
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), # why do we need this ?
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)# per tocken tranforamtion
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # x = x + self.sa() # this adding x is residual ( so we create a direct shortcut back to input)
        # x = x + self.ffwd(x)
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) # n_embed is number of embedding per ovcabn
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # # # self.sa_head = Head(n_embd) # this allows some talking 
        # # self.sa_head = MultiHeadAttention(4, n_embd//4)# this allows 4 channels of talking so i can ask for 4 things 
        # # self.ffwd = FeedFoward(n_embd)
        # self.blocks = nn.Sequential( # this is used to use a block of sa and feed forward and use it multiple times.
        #     Block(n_embd, n_head=4),
        #     Block(n_embd, n_head=4),
        #     Block(n_embd, n_head=4),
        # )
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm 
        # before layernorm was applied after a transofrmation but now its applied before a transformation
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape
        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C) # this embeds the word or vocab part
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # this embeds the position of the word or vocab.
        x = tok_emb + pos_emb
        # x = self.sa_head(x) # all tokens are comminicating to find important information 
        # x = self.ffwd(x) # now for each token we give them time to think of what they have found 
        x = self.blocks(x)
        x = self.ln_f(x) 
        logits = self.lm_head(x) # (B,T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size_tokens 
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))